# SNN Regression for Two Outputs

Notebook baseado no fluxo de `SNN_Regression_pendulum.ipynb`, mas configurado para treinar a rede com duas saídas: ângulo e posição.

In [ ]:
from pathlib import Path
import random
import sys

import numpy as np
import torch
from spikingjelly.activation_based import surrogate

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT / "SNN-regression-main"]:
    if (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import (
    SNN_Net,
    create_dataloaders,
    layer_list_sew,
    plot_all,
    read_pencil_file,
    test,
    train,
)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Ajuste estes caminhos para os seus arquivos reais.
EVENT_FILE = PROJECT_ROOT / "data" / "hough_camp1_cam1.aedat4"
LABEL_FILE = PROJECT_ROOT / "data" / "hough_cam1_cam1_hough.csv"

events_per_frame, labels, label_metadata = read_pencil_file(
    EVENT_FILE,
    LABEL_FILE,
    time_window=3000,
    START_FRAME=300,
    END_FRAME=-1,
    timestamp_column="timestamp_us",
    angle_column="lin_m",
    position_column="lin_b",
    angle_unit="rad",
    position_scale=1.0,
    position_name="x",
    position_unit="mm",
)

labels.shape, label_metadata

In [ ]:
trainloader, valloader, testloader = create_dataloaders(
    events_per_frame,
    labels,
    test_ratio=0.05,
    val_ratio=0.07,
    SEQ_LENGTH=2000,
    BATCH_SIZE=4,
)

CONFIG = {
    "experiment": "Pencil",
    "event_file": str(EVENT_FILE),
    "label_file": str(LABEL_FILE),
    "time_window": label_metadata["time_window_us"],
    "output_dim": 2,
    "target_specs": [
        {
            "name": "angle",
            "display_name": "Angle",
            "unit": "deg",
            "normalize": "angle_pm_pi",
        },
        {
            "name": label_metadata["position_name"],
            "display_name": label_metadata["position_name"],
            "unit": label_metadata["position_unit"],
            "normalize": "linear",
            "min": label_metadata["position_min"],
            "max": label_metadata["position_max"],
        },
    ],
    "block_type": "SEW",
    "hidden": 256,
    "reset_type": "soft",
    "tau": 2.0,
    "final_tau": 20.0,
    "surrogate_function": "ATan",
    "Plif": False,
    "norm_type": "BN",
    "learnable_norm": True,
    "init_scale": 5.0,
    "K": 10,
    "true_value_initialization": False,
    "transient": 200,
    "batch_size": 4,
    "sequence_length": 2000,
    "optimizer": "SGD",
    "learning_rate": 1e-2,
    "momentum": 0.9,
    "weight_decay": 0.0,
    "scheduler": "ReduceLROnPlateau",
    "scheduler_factor": 0.5,
    "scheduler_patience": 1,
    "min_lr": 1e-6,
    "num_epochs": 30,
    "device": str(device),
    "early_stop_patience": 10,
}

model = SNN_Net(
    tau=CONFIG["tau"],
    final_tau=CONFIG["final_tau"],
    layer_list=layer_list_sew,
    hidden=CONFIG["hidden"],
    v_reset=None,
    surrogate_function=surrogate.ATan(),
    connect_f="ADD",
    Plif=CONFIG["Plif"],
    norm_type=CONFIG["norm_type"],
    learnable_norm=CONFIG["learnable_norm"],
    init_scale=CONFIG["init_scale"],
    output_dim=CONFIG["output_dim"],
).to(device)

model

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "models" / "model_SEW_BN_SGD" / "checkpoints_pencil"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train(
    model,
    trainloader,
    valloader,
    CONFIG,
    OUTPUT_DIR,
    loss_fn=torch.nn.MSELoss(),
    use_wandb=False,
    project_name="snn-pencil-regression",
)

In [ ]:
model.load_state_dict(torch.load(OUTPUT_DIR / "best_model_weights.pth", map_location=device, weights_only=True))
results = test(model, testloader, CONFIG, monitor_mode="norm", loss_fn=torch.nn.MSELoss())
plot_all(results, window_start=200, window_end=-1, experiment_type="Pencil")